In [ ]:
from pathlib import Path

RUN = "runs/tactic-temporal"
# Optional control run with the random-thirds model; set to None to leave it out.
CONTROL_RUN = "runs/tactic-temporal-random-thirds"
# Optional runs on detector position features instead of image features; None leaves them out.
POSITION_RUN = "runs/tactic-positions"
POSITION_ABLATION_RUN = "runs/tactic-positions-no-bomb"  # Same, without the bomb-badge channel.
# Optional: the same image models under the TCN's epoch rule (--epoch-mode other-folds),
# and a train-tactics cross-validation run (no --evaluate-test). None leaves them out.
OTHER_FOLDS_RUN = "runs/tactic-temporal-other-folds"
TCN_RUN = "runs/tactic-tcn"

# Works when the kernel starts in the repository root or in notebooks/.
ROOT = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "stratscout").is_dir() and (p / "AGENTS.md").is_file()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the CV-DL repository or its notebooks folder.")

run_dir = (ROOT / RUN).resolve()
if not (run_dir / "summary.json").is_file():
    raise FileNotFoundError(
        f"No temporal run in: {run_dir}\n"
        "Change RUN to an existing run, for example 'runs/tactic-temporal-20261116', "
        "or create the default run.\n"
        "From the repository root, with the .venv environment active, run:\n"
        "python -m stratscout tactic-temporal "
        "--manifest data/processed/datasets/2026-11-16/train/manifest.csv "
        f'--output "{RUN}" --exclude-uncertain --shuffles 20'
    )
print(f"Reading saved results from {run_dir}")
control_dir = (ROOT / CONTROL_RUN).resolve() if CONTROL_RUN else None
if control_dir is not None and not (control_dir / "random-thirds" / "summary.json").is_file():
    print(f"No random-thirds control in {control_dir}; it is left out.")
    print(
        "To add it, run tactic-temporal with --models random-thirds and "
        f'--features "{RUN}/frame-features.pt" --shuffles 20 into that folder.'
    )
    control_dir = None
elif control_dir is not None:
    print(f"Reading the random-thirds control from {control_dir}")

# Tactic model comparison: baseline, temporal average pooling, GRU and TCN

1. Select the project's **.venv** Python kernel. If it is missing, install `ipykernel` in that environment.
2. Set `RUN` in the first cell to an existing `tactic-temporal` run, then run the cells from top to bottom.
3. Keep the figures on screen; this notebook does not write images, change results or train a model.

The run must contain `mean` (the Plan A baseline, refitted on the same frame features), `thirds` and `gru`.
`thirds` is **temporal average pooling (3 bins) + linear**: average the early, middle and late frame features separately, concatenate the three vectors, then apply a linear classifier. It preserves coarse order, but loses order within each bin and does not receive elapsed time. The short keys and folder names stay unchanged for compatibility.
`CONTROL_RUN` optionally adds `random-thirds`: thirds' three averages and head, but with frames put into the groups at random.
If it scores like `mean` and below `thirds`, that is evidence that time order helps on these development matches (not proof); if it scores like `thirds`, the gain comes only from having more inputs.
All models share the frames, whole-match folds, epoch choice, seeds, class weights and metrics, so only the model differs.
The notebook reads saved JSON files only. It never opens clips, checkpoints or the test split.

**Read every figure with this in mind:** there are about 80 clips from four matches. A small gap between a model and the baseline is within noise. Comparing a gap with the seed spread is a rough guide, not a significance test: the seeds rerun the same four matches. These are development scores, not test results.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import MaxNLocator

plt.switch_backend("module://matplotlib_inline.backend_inline")
plt.rcParams.update(
    {
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "legend.fontsize": 9,
        "figure.dpi": 110,
    }
)

summary = json.loads((run_dir / "summary.json").read_text(encoding="utf-8"))
if not {"mean", "thirds", "gru"} <= set(summary["models"]):
    raise ValueError("This notebook needs a run with the mean, thirds and gru models.")
entries = dict(summary["models"])
folders = {model: run_dir / model for model in ("mean", "thirds", "gru")}
if control_dir is not None:
    control = json.loads((control_dir / "summary.json").read_text(encoding="utf-8"))
    entries["random-thirds"] = control["models"]["random-thirds"]
    folders["random-thirds"] = control_dir / "random-thirds"
models = [m for m in ("mean", "thirds", "random-thirds", "gru") if m in folders]
labels = {
    "mean": "Baseline (mean)",
    "thirds": "Temporal average pooling (3 bins) + linear",
    "random-thirds": "Random-bin average pooling + linear (control)",
    "gru": "GRU",
}
plot_labels = {**labels,
    "thirds": "Temporal average\npooling (3 bins)\n+ linear",
    "random-thirds": "Random-bin average\npooling + linear\n(control)",
}
colours = {"mean": "#555555", "thirds": "#0072B2", "random-thirds": "#56B4E9", "gru": "#D55E00"}
width = 0.8 / len(models)


def offsets(count):
    return (np.arange(count) - (len(models) - 1) / 2) * width


configs, folds = {}, {}
for model in models:
    folder = folders[model]
    configs[model] = json.loads((folder / "config.json").read_text(encoding="utf-8"))
    if configs[model].get("split") != "train":
        raise ValueError("Every model must come from the training manifest.")
    folds[model] = {}
    for path in sorted(folder.glob("fold-*.json")):
        if path.stem.removeprefix("fold-").isdigit():
            saved = json.loads(path.read_text(encoding="utf-8"))
            folds[model][saved["fold"]] = saved
numbers = sorted(folds["mean"])
for model in models[1:]:
    same = sorted(folds[model]) == numbers and all(
        folds[model][n]["validation_clip_ids"] == folds["mean"][n]["validation_clip_ids"]
        and folds[model][n]["train_clip_ids"] == folds["mean"][n]["train_clip_ids"]
        for n in numbers
    )
    if not same or configs[model]["settings"] != configs["mean"]["settings"]:
        raise ValueError(f"{model} was not trained on the baseline's folds and settings.")

mode = summary["epoch_mode"]
classes = configs["mean"]["classes"]
class_names = [label.replace("_", " ").capitalize() for label in classes]
seeds = configs["mean"]["settings"]["seeds"]
samples = json.loads((run_dir / "samples.json").read_text(encoding="utf-8"))
match_folders = {sample["match_id"]: sample["match_folder"] for sample in samples}
names = {
    "furia": "Furia vs Falcons",
    "spirit": "Spirit vs Falcons",
    "aurora": "Aurora vs Vitality",
    "mouz": "MOUZ vs BetBoom",
}
fold_names = {
    n: names.get(
        match_folders[folds["mean"][n]["validation_match_id"]].split("-")[0],
        match_folders[folds["mean"][n]["validation_match_id"]].replace("-", " ").title(),
    )
    for n in numbers
}


def run_of(model, number, seed):
    return next(r for r in folds[model][number]["runs"] if r["seed"] == seed)


def score(model, number, seed, key="macro_f1"):
    return run_of(model, number, seed)["metrics"][key]


def overall(model, seed, key="macro_f1"):
    """Equal average over validation matches, as in the reports."""
    return float(np.mean([score(model, n, seed, key) for n in numbers]))


def class_f1(model, number, label):
    return float(np.mean([run_of(model, number, s)["metrics"]["per_class"][label]["f1"] for s in seeds]))


print("Epoch choice:", mode)
print("Validation matches:", ", ".join(fold_names.values()))
print("Seeds:", seeds)
print("Clips:", len(samples), "| excluded:", configs["mean"]["excluded_clip_ids"] or "none")
print("Trainable weights:", {m: entries[m]["trainable_parameters"] for m in models})
print(f"{'model':<48}{'macro-F1':>10}{'SD':>8}{'accuracy':>10}{'optimistic':>12}")
for model in models:
    entry = entries[model]
    print(
        f"{labels[model]:<48}{entry['macro_f1']['mean']:>10.3f}{entry['macro_f1']['std']:>8.3f}"
        f"{entry['accuracy']['mean']:>10.3f}{entry['optimistic_macro_f1']['mean']:>12.3f}"
    )
majority = entries["mean"]["majority"]
print(f"{'Majority guess':<48}{majority['macro_f1']:>10.3f}{'':>8}{majority['accuracy']:>10.3f}")

## Overall score

Each dot is one seed's macro-F1, averaged equally over the validation matches; the bar is the mean of the dots. The dashed line is the majority-class guess. Compare the gap between two bars with the spread of the dots: a gap about the size of that spread is probably not a real difference. This is a rough guide, not a significance test: the seeds rerun the same four matches and clips, so the dots show training noise, not how scores would move on other matches.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), layout="constrained")
for position, model in enumerate(models):
    values = [overall(model, seed) for seed in seeds]
    ax.bar(position, np.mean(values), 0.6, color=colours[model], alpha=0.75)
    ax.scatter([position] * len(values), values, color="black", zorder=3, s=28, label="One seed" if position == 0 else None)
    ax.annotate(f"{np.mean(values):.3f}", (position, max(values)), xytext=(0, 8), textcoords="offset points", ha="center")
ax.axhline(majority["macro_f1"], linestyle="--", color="#999999", label=f"Majority guess ({majority['macro_f1']:.3f})")
ax.set_xticks(range(len(models)), [plot_labels[m] for m in models])
ax.set_ylim(0, 1)
ax.set_ylabel("Macro-F1, mean over validation matches")
ax.set_title(f"Overall {mode} macro-F1 by model")
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)
ax.legend(loc="upper left")
plt.show()
plt.close(fig)

## Each validation match

Bars are the mean over seeds; the black lines run from the lowest to the highest seed. The number under each match is its validation clips. A model that is ahead on one match and behind on another has not shown a consistent gain.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.5), layout="constrained")
for offset, model in zip(offsets(len(models)), models, strict=True):
    positions = np.arange(len(numbers)) + offset
    values = np.array([[score(model, n, seed) for seed in seeds] for n in numbers])
    means = values.mean(axis=1)
    bars = ax.bar(positions, means, width, color=colours[model], alpha=0.8, label=plot_labels[model])
    ax.vlines(positions, values.min(axis=1), values.max(axis=1), color="black", linewidth=1.4)
    ax.bar_label(bars, labels=[f"{m:.2f}" for m in means], padding=14, fontsize=8)
majorities = [folds["mean"][n]["majority"]["macro_f1"] for n in numbers]
ax.hlines(majorities, np.arange(len(numbers)) - 0.42, np.arange(len(numbers)) + 0.42, color="#999999", linestyle="--", label="Majority guess")
ax.set_xticks(
    range(len(numbers)),
    [f"{fold_names[n]}\n(n={len(folds['mean'][n]['validation_clip_ids'])})" for n in numbers],
)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Validation macro-F1")
ax.set_title("Macro-F1 for each validation match: mean and range over seeds")
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)
ax.legend(loc="upper right", ncols=2)
plt.show()
plt.close(fig)

## Difference from the baseline, seed by seed

Each dot is one validation match and seed: the model's macro-F1 minus the baseline's on exactly the same clips and seed. Dots on both sides of zero mean the model is not reliably better or worse. The diamond is the mean of the dots.

Compare thirds with random thirds here: both have the same inputs and head, so a gap between them points to frame order. Look also at how much of the thirds gain comes from one match (MOUZ vs BetBoom).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5), layout="constrained")
markers = ["o", "s", "^", "v", "P", "X"]
for position, model in enumerate(models[1:]):
    differences = []
    for index, n in enumerate(numbers):
        values = [score(model, n, seed) - score("mean", n, seed) for seed in seeds]
        differences += values
        jitter = position + np.linspace(-0.18, 0.18, len(numbers))[index]
        ax.scatter([jitter] * len(values), values, marker=markers[index % len(markers)], color=colours[model], edgecolor="black", s=45, label=fold_names[n] if position == 0 else None)
    ax.scatter(position + 0.32, np.mean(differences), marker="D", color="white", edgecolor="black", s=80, zorder=4, label="Mean of the dots" if position == 0 else None)
    ax.annotate(f"{np.mean(differences):+.3f}", (position + 0.32, np.mean(differences)), xytext=(10, -4), textcoords="offset points")
ax.axhline(0, color="#333333", linewidth=1.2)
ax.set_xticks(range(len(models) - 1), [plot_labels[m] for m in models[1:]])
ax.set_xlim(-0.5, len(models) - 1.2)
ax.set_ylabel("Macro-F1 minus the baseline's")
ax.set_title("Paired difference from the baseline, per validation match and seed")
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)
ax.legend(loc="best", ncols=2)
plt.show()
plt.close(fig)

## Each class, averaged over the validation matches

Bars average F1 over the validation matches and seeds. The number under each class is its total validation clips across matches.

Read the table above as the answer to one question: **does the thirds gain come from frame order, or just from having three times as many inputs?** Random thirds has the extra inputs without the order. If it lands near the baseline and well below thirds, that is evidence that time order helps on these development matches, not proof.

In [ ]:
if "random-thirds" not in models:
    print("No random-thirds control loaded; set CONTROL_RUN to compare frame order with input size.")
else:

    def paired(a, b):
        return np.array([[score(a, n, s) - score(b, n, s) for s in seeds] for n in numbers])

    print(f"{'Comparison':<90}{'mean gain':>10}{'pairs ahead':>13}{'without MOUZ':>14}")
    mouz = [i for i, n in enumerate(numbers) if fold_names[n].startswith("MOUZ")]
    for a, b in (("thirds", "mean"), ("random-thirds", "mean"), ("thirds", "random-thirds")):
        d = paired(a, b)
        rest = np.delete(d, mouz, axis=0) if mouz else d
        print(
            f"{labels[a] + ' - ' + labels[b]:<90}{d.mean():>+10.3f}"
            f"{f'{int((d > 0).sum())}/{d.size}':>13}{rest.mean():>+14.3f}"
        )
    print("\nPer validation match (mean over seeds):")
    for i, n in enumerate(numbers):
        print(
            f"  {fold_names[n]:<20}" + "".join(
                f"{labels[a]} - {labels[b]}: {paired(a, b)[i].mean():+.3f}   "
                for a, b in (("thirds", "mean"), ("thirds", "random-thirds"))
            )
        )

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.5), layout="constrained")
for offset, model in zip(offsets(len(models)), models, strict=True):
    values = [np.mean([class_f1(model, n, label) for n in numbers]) for label in classes]
    bars = ax.bar(np.arange(len(classes)) + offset, values, width, color=colours[model], alpha=0.8, label=plot_labels[model])
    ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
totals = [sum(folds["mean"][n]["validation_counts"][label] for n in numbers) for label in classes]
ax.set_xticks(range(len(classes)), [f"{name}\n(n={total})" for name, total in zip(class_names, totals, strict=True)])
ax.set_ylim(0, 1.1)
ax.set_ylabel("Mean F1")
ax.set_title("Per-class F1, mean over validation matches and seeds")
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)
ax.legend(loc="upper left", ncols=2)
plt.show()
plt.close(fig)

## Each class, one figure per validation match

The number under each class is that match's validation clips. With one to three clips, a single prediction moves F1 a long way, so these bars are the noisiest in the notebook.

In [ ]:
for n in numbers:
    fig, ax = plt.subplots(figsize=(12, 5), layout="constrained")
    for offset, model in zip(offsets(len(models)), models, strict=True):
        values = [class_f1(model, n, label) for label in classes]
        bars = ax.bar(np.arange(len(classes)) + offset, values, width, color=colours[model], alpha=0.8, label=plot_labels[model])
        ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
    counts = folds["mean"][n]["validation_counts"]
    ax.set_xticks(range(len(classes)), [f"{name}\n(n={counts[label]})" for label, name in zip(classes, class_names, strict=True)])
    ax.set_ylim(0, 1.15)
    ax.set_ylabel("Mean F1 over seeds")
    ax.set_title(f"{fold_names[n]}: per-class F1")
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)
    ax.legend(loc="upper left", ncols=2)
    plt.show()
    plt.close(fig)

## Confusion matrices, summed over validation matches and seeds

Rows are true classes; columns are predicted classes. Each clip is counted once per seed, so these are prediction counts, not extra clips. Look at which classes each model confuses, for example direct against split on the same site.

In [ ]:
columns = 2
rows = -(-len(models) // columns)
fig, axes = plt.subplots(rows, columns, figsize=(13, 5.8 * rows), layout="constrained")
for ax in axes.flat[len(models):]:
    ax.set_visible(False)
for ax, model in zip(axes.flat, models):
    matrix = np.sum(
        [run["metrics"]["confusion_matrix"] for n in numbers for run in folds[model][n]["runs"]],
        axis=0,
    )
    ax.imshow(matrix, cmap="Blues", vmin=0)
    ax.set_xticks(range(len(classes)), class_names, rotation=30, ha="right")
    ax.set_yticks(range(len(classes)), class_names)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title(f"{plot_labels[model]}: {matrix.sum()} predictions")
    for row in range(len(classes)):
        for column in range(len(classes)):
            ax.text(
                column,
                row,
                str(int(matrix[row, column])),
                ha="center",
                va="center",
                fontsize=12,
                color="white" if matrix[row, column] > matrix.max() / 2 else "black",
            )
plt.show()
plt.close(fig)

## Shuffled-label check

Each small dot is one refit of the whole procedure on labels shuffled within each match, scored against those shuffled labels. The diamond is the real score. A sound procedure puts the shuffled dots near chance and the real score well above all of them. If so, the check found no sign of leakage. It cannot rule out every leak, and it does not show how the model will do on a new match.

In [ ]:
checks = {}
for model in models:
    path = folders[model] / "shuffled-labels.json"
    if path.is_file():
        checks[model] = json.loads(path.read_text(encoding="utf-8"))
if not checks:
    print("This run has no shuffled-label check. Run tactic-temporal with --shuffles 20 to add one.")
else:
    fig, ax = plt.subplots(figsize=(10, 5), layout="constrained")
    for position, (model, check) in enumerate(checks.items()):
        values = [row["macro_f1"] for row in check["permutations"]]
        spread = np.linspace(-0.15, 0.15, len(values))
        ax.scatter(position + spread, values, color=colours[model], alpha=0.6, s=24, label="Shuffled labels" if position == 0 else None)
        ax.scatter(position, check["real"]["macro_f1"], marker="D", color=colours[model], edgecolor="black", s=90, zorder=4, label="Real labels" if position == 0 else None)
        ax.annotate(f"real {check['real']['macro_f1']:.3f}\nshuffled {np.mean(values):.3f}\n{check['at_least_real']} of {check['count']} reach real", (position, 0.86), ha="center", fontsize=9)
    ax.set_xticks(range(len(checks)), [plot_labels[m] for m in checks])
    ax.set_xlim(-0.6, len(checks) - 0.4)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Macro-F1, mean over validation matches and seeds")
    ax.set_title("Real labels against labels shuffled within each match")
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)
    ax.legend(loc="center right")
    plt.show()
    plt.close(fig)

## Image features against position features

The same models (`mean`, `thirds`, `gru`) trained on two kinds of per-frame input, with the same clips, frames, folds, epoch choice, seeds and class weights:

- **image features**: 512 frozen ResNet-18 numbers per frame (the run above);
- **position features**: 260 numbers per frame from the minimap detector. An 8x8 grid over the radar holds the visible T and CT icons, the inferred bomb carrier and the bomb badge, each in its own channel, plus four totals. No tracker identity; missing players are not filled in.

`POSITION_RUN` points at the position run. If it is missing, this section says so and stops. `POSITION_ABLATION_RUN` is an optional extra run without the bomb-badge channel, used to show what that channel adds.

In [ ]:
def load_run(folder, needed):
    """Fold files of one run folder's models, checked against the image run's folds."""
    loaded = {}
    for model in needed:
        model_dir = folder / model
        if not (model_dir / "summary.json").is_file():
            return None
        model_folds = {}
        for path in sorted(model_dir.glob("fold-*.json")):
            if path.stem.removeprefix("fold-").isdigit():
                saved = json.loads(path.read_text(encoding="utf-8"))
                model_folds[saved["fold"]] = saved
        if sorted(model_folds) != numbers or any(
            model_folds[n]["validation_clip_ids"] != folds["mean"][n]["validation_clip_ids"]
            or model_folds[n]["train_clip_ids"] != folds["mean"][n]["train_clip_ids"]
            for n in numbers
        ):
            raise ValueError(f"{model_dir} does not use the image run's folds and clips.")
        config = json.loads((model_dir / "config.json").read_text(encoding="utf-8"))
        if config.get("split") != "train" or config["settings"] != configs["mean"]["settings"]:
            raise ValueError(f"{model_dir} was not trained with the image run's settings.")
        loaded[model] = {"folds": model_folds, "config": config, "dir": model_dir}
    return loaded


base_models = ["mean", "thirds", "gru"]
position_dir = (ROOT / POSITION_RUN).resolve() if POSITION_RUN else None
positions = load_run(position_dir, base_models) if position_dir and position_dir.is_dir() else None
ablation_dir = (ROOT / POSITION_ABLATION_RUN).resolve() if POSITION_ABLATION_RUN else None
ablation = None
if positions is not None and ablation_dir is not None and ablation_dir.is_dir():
    ablation = load_run(ablation_dir, base_models)
if positions is None:
    print(f"No position run at {position_dir}; this section is skipped.")
    print(
        "To create it, run tactic-temporal with --inputs positions --shuffles 20 "
        f'and --output "{POSITION_RUN}".'
    )
else:
    print(f"Reading position features from {position_dir}")
    print("Width per frame:", positions["mean"]["config"]["inputs"]["width"])
    print("Bomb-badge ablation:", ablation_dir if ablation else "not found; left out")
    sources = {
        ("images", m): {"folds": folds[m], "dir": folders[m]} for m in base_models
    } | {("positions", m): positions[m] for m in base_models}
    if ablation:
        sources |= {("no bomb badge", m): ablation[m] for m in base_models}

    def scores(source, model, n, key="macro_f1"):
        runs = {r["seed"]: r for r in sources[(source, model)]["folds"][n]["runs"]}
        return [runs[s]["metrics"][key] for s in seeds]

    def overall_scores(source, model, key="macro_f1"):
        """Per seed: equal average over validation matches."""
        return np.mean([scores(source, model, n, key) for n in numbers], axis=0)

    def per_class(source, model, label):
        return float(
            np.mean(
                [
                    r["metrics"]["per_class"][label]["f1"]
                    for n in numbers
                    for r in sources[(source, model)]["folds"][n]["runs"]
                ]
            )
        )

    print(f"\n{'Model':<8}{'Inputs':<16}{'macro-F1':>10}{'SD':>7}{'accuracy':>10}")
    for model in base_models:
        for source in ("images", "positions", "no bomb badge"):
            if (source, model) in sources:
                f1, acc = overall_scores(source, model), overall_scores(source, model, "accuracy")
                print(f"{model:<8}{source:<16}{f1.mean():>10.3f}{f1.std():>7.3f}{acc.mean():>10.3f}")

### Overall and per match

Bars are the mean over seeds; the black lines run from the lowest to the highest seed. Solid bars are image features, hatched bars position features. A gap no bigger than the seed range, or one that flips between matches, is not a real difference.

In [ ]:
if positions is None:
    print("Skipped: no position run.")
else:
    fig, axes = plt.subplots(1, len(numbers) + 1, figsize=(19, 5), sharey=True, layout="constrained")
    groups = [("Overall", None)] + [(fold_names[n], n) for n in numbers]
    bar = 0.38
    for ax, (title, n) in zip(axes, groups, strict=True):
        for position, model in enumerate(base_models):
            for side, source, hatch in ((-1, "images", None), (1, "positions", "//")):
                values = overall_scores(source, model) if n is None else np.array(scores(source, model, n))
                x = position + side * bar / 2
                ax.bar(x, values.mean(), bar, color=colours[model], alpha=0.85 if hatch is None else 0.45, hatch=hatch, edgecolor="black", linewidth=0.5)
                ax.vlines(x, values.min(), values.max(), color="black", linewidth=1.3)
                ax.annotate(f"{values.mean():.2f}", (x, values.max()), xytext=(0, 4), textcoords="offset points", ha="center", fontsize=8)
        ax.set_xticks(range(len(base_models)), [plot_labels[m] for m in base_models])
        ax.set_title(title if n is None else f"{title} (n={len(folds['mean'][n]['validation_clip_ids'])})")
        ax.set_ylim(0, 1)
        ax.grid(axis="y", alpha=0.2)
        ax.set_axisbelow(True)
    axes[0].set_ylabel("Macro-F1")
    from matplotlib.patches import Patch

    axes[-1].legend(
        handles=[Patch(facecolor="#999999", edgecolor="black", label="Image features"), Patch(facecolor="#dddddd", edgecolor="black", hatch="//", label="Position features")],
        loc="upper right",
    )
    fig.suptitle("Image against position features: macro-F1, mean and range over seeds", fontsize=14)
    plt.show()
    plt.close(fig)

### Per class: did direct against split improve, or only retake and A against B?

F1 per class, averaged over the validation matches and seeds. The table also shows how often each true split was predicted as the direct attack on the same site (and the reverse), summed over matches and seeds. If position features only help RETAKE or the A/B site, the split/direct columns stay where they were.

In [ ]:
if positions is None:
    print("Skipped: no position run.")
else:
    pairs = {"A_SPLIT": "A_DIRECT", "A_DIRECT": "A_SPLIT", "B_SPLIT": "B_DIRECT", "B_DIRECT": "B_SPLIT"}
    columns = [(m, s) for m in base_models for s in ("images", "positions")]
    print(f"{'Class':<10}{'clips':>6}" + "".join(f"{m[:6] + '/' + s[:3]:>12}" for m, s in columns))
    for label in classes:
        clips = sum(folds["mean"][n]["validation_counts"][label] for n in numbers)
        print(f"{label:<10}{clips:>6}" + "".join(f"{per_class(s, m, label):>12.3f}" for m, s in columns))

    def confused(source, model, truth, predicted):
        i, j = classes.index(truth), classes.index(predicted)
        return int(sum(r["metrics"]["confusion_matrix"][i][j] for n in numbers for r in sources[(source, model)]["folds"][n]["runs"]))

    print("\nSplit/direct mix-ups on the same site (predictions, summed over matches and seeds):")
    print(f"{'true -> predicted':<24}" + "".join(f"{m[:6] + '/' + s[:3]:>12}" for m, s in columns))
    for truth, predicted in pairs.items():
        print(f"{truth + ' -> ' + predicted:<24}" + "".join(f"{confused(s, m, truth, predicted):>12}" for m, s in columns))

    fig, axes = plt.subplots(1, len(base_models), figsize=(18, 5), sharey=True, layout="constrained")
    for ax, model in zip(axes, base_models, strict=True):
        x = np.arange(len(classes))
        for side, source, hatch in ((-1, "images", None), (1, "positions", "//")):
            values = [per_class(source, model, label) for label in classes]
            bars = ax.bar(x + side * 0.2, values, 0.4, color=colours[model], alpha=0.85 if hatch is None else 0.45, hatch=hatch, edgecolor="black", linewidth=0.5, label="Image features" if hatch is None else "Position features")
            ax.bar_label(bars, fmt="%.2f", padding=2, fontsize=8)
        ax.set_xticks(x, class_names, rotation=20)
        ax.set_title(plot_labels[model])
        ax.set_ylim(0, 1.1)
        ax.grid(axis="y", alpha=0.2)
        ax.set_axisbelow(True)
    axes[0].set_ylabel("Mean F1")
    axes[0].legend(loc="upper left")
    fig.suptitle("Per-class F1: image against position features", fontsize=14)
    plt.show()
    plt.close(fig)

### Did the linear models fit the sparse position inputs?

The final training loss of the refit that was scored, averaged over matches and seeds. Training loss uses class weights, so a model that learned nothing sits near ln 5 = 1.61. A loss far below that means the sparse inputs did not stop the linear head from fitting the training clips.

In [ ]:
if positions is None:
    print("Skipped: no position run.")
else:

    def final_losses(source, model):
        folder = sources[(source, model)]["dir"]
        values, epochs = [], []
        for n in numbers:
            for s in seeds:
                history = json.loads((folder / f"fold-{n}-seed-{s}-history.json").read_text(encoding="utf-8"))
                values.append(history[-1]["train_loss"])
                epochs.append(history[-1]["epoch"])
        return np.array(values), np.array(epochs)

    print(f"{'Model':<8}{'Inputs':<12}{'final train loss':>18}{'range':>16}{'scored epoch':>14}")
    for model in base_models:
        for source in ("images", "positions"):
            loss, epochs = final_losses(source, model)
            print(f"{model:<8}{source:<12}{loss.mean():>18.3f}{f'{loss.min():.2f}-{loss.max():.2f}':>16}{np.median(epochs):>14.0f}")
    print(f"\nNo learning (uniform guess): {np.log(len(classes)):.3f}")

### What the bomb-badge channel adds

The ablation run is the same position run with the bomb-badge channel and its total removed, refitted with the same settings (no shuffles). The difference is that channel's contribution on these matches. It is one extra run planned before the scores were seen.

In [ ]:
if positions is None:
    print("Skipped: no position run.")
elif ablation is None:
    print("No bomb-badge ablation run found, so its contribution is not separated here.")
else:
    print(f"{'Model':<8}{'with badge':>12}{'without':>10}{'difference':>12}   per class (with - without)")
    for model in base_models:
        with_badge, without = overall_scores("positions", model), overall_scores("no bomb badge", model)
        deltas = "  ".join(f"{label}: {per_class('positions', model, label) - per_class('no bomb badge', model, label):+.2f}" for label in classes)
        print(f"{model:<8}{with_badge.mean():>12.3f}{without.mean():>10.3f}{with_badge.mean() - without.mean():>+12.3f}   {deltas}")

### Shuffled-label check for the position features

As above: each small dot is one refit on labels shuffled within each match, the diamond is the real score. The real score should sit well above all the shuffled dots; if it does, the check found no sign of leakage.

In [ ]:
if positions is None:
    print("Skipped: no position run.")
else:
    fig, ax = plt.subplots(figsize=(11, 5), layout="constrained")
    position_x = 0
    ticks = []
    for model in base_models:
        for source in ("images", "positions"):
            path = sources[(source, model)]["dir"] / "shuffled-labels.json"
            if not path.is_file():
                continue
            check = json.loads(path.read_text(encoding="utf-8"))
            values = [row["macro_f1"] for row in check["permutations"]]
            spread = np.linspace(-0.15, 0.15, len(values))
            hatch_alpha = 0.6 if source == "images" else 0.3
            ax.scatter(position_x + spread, values, color=colours[model], alpha=hatch_alpha, s=22)
            ax.scatter(position_x, check["real"]["macro_f1"], marker="D", color=colours[model], edgecolor="black", s=80, zorder=4)
            ax.annotate(f"{check['real']['macro_f1']:.2f}\n{check['at_least_real']}/{check['count']}", (position_x, check["real"]["macro_f1"]), xytext=(14, -6), textcoords="offset points", fontsize=8)
            ticks.append((position_x, f"{plot_labels[model]}\n{source}"))
            position_x += 1
    ax.set_xticks([t for t, _ in ticks], [label for _, label in ticks])
    ax.set_ylim(0, 1)
    ax.set_ylabel("Macro-F1, mean over matches and seeds")
    ax.set_title("Real labels (diamonds) against shuffled labels (dots); x/20 = shuffles at or above real")
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)
    plt.show()
    plt.close(fig)

## All four models under the TCN's protocol

The TCN (`train-tactics`) chooses its epoch differently from the models above, so its score cannot be set beside theirs directly. `OTHER_FOLDS_RUN` refits the baseline, thirds and GRU with the TCN's rule (`--epoch-mode other-folds`); nothing else changes:

- every validation match is fitted for all epochs, and each match's curve is averaged over seeds;
- a match is scored at the epoch that is best on the mean curve of the **other** matches (earliest on ties), the same epoch for every seed.

**This rule is slightly optimistic.** The other matches' models were trained on the match being scored, so that match helps choose its own epoch indirectly. The nested numbers above keep it out completely, so they are the **stricter estimate**.

Two numbers are shown, as the TCN reports them:

- **mean over seeds**: each seed's macro-F1, averaged equally over the four matches, then over seeds;
- **seed ensemble**: each clip's class probabilities averaged over the three seeds, then scored once over all 82 clips pooled together.

Per class and site-against-split use the seed ensemble, pooled over the 82 clips, for all four models. Remaining differences: the TCN reads detector features, trains in mini-batches with jittered and dropped detections, and scores every 10th epoch; the ResNet models train full-batch with no augmentation and score every epoch.

In [ ]:
import csv

image_models = ["mean", "thirds", "gru"]


def match_of(clip_id):
    return clip_id.split(".")[0]


def ensemble_from_rows(rows):
    """Seed ensemble pooled over clips: (macro-F1, accuracy, per-class F1, confusion)."""
    by_clip = {}
    for row in rows:
        clip = by_clip.setdefault(row["clip_id"], {"truth": row["truth"], "probs": []})
        clip["probs"].append([float(row[f"p_{c}"]) for c in classes])
    truth = np.array([classes.index(c["truth"]) for c in by_clip.values()])
    predicted = np.array([np.mean(c["probs"], axis=0).argmax() for c in by_clip.values()])
    matrix = np.zeros((len(classes), len(classes)), int)
    np.add.at(matrix, (truth, predicted), 1)
    f1 = {}
    for i, label in enumerate(classes):
        denominator = matrix[i].sum() + matrix[:, i].sum()
        f1[label] = 2 * matrix[i, i] / denominator if denominator else 0.0
    return {
        "macro_f1": float(np.mean(list(f1.values()))),
        "accuracy": float(np.trace(matrix) / matrix.sum()),
        "per_class": f1,
        "confusion": matrix,
        "clips": {c: match_of(c) for c in by_clip},
    }


def read_image_run(folder, model):
    model_dir = folder / model
    summary = json.loads((model_dir / "summary.json").read_text(encoding="utf-8"))
    fold_files = [json.loads(p.read_text(encoding="utf-8")) for p in sorted(model_dir.glob("fold-[0-9].json"))]
    with (model_dir / "predictions.csv").open(encoding="utf-8") as stream:
        rows = list(csv.DictReader(stream))
    per_match = {
        match_of(f["validation_clip_ids"][0]): float(np.mean([r["metrics"]["macro_f1"] for r in f["runs"]]))
        for f in fold_files
    }
    shuffled_path = model_dir / "shuffled-labels.json"
    return {
        "mode": summary["epoch_mode"],
        "seed_mean": summary["model"]["macro_f1"]["mean"],
        "seed_sd": summary["model"]["macro_f1"]["std"],
        "accuracy": summary["model"]["accuracy"]["mean"],
        "per_match": per_match,
        "ensemble": ensemble_from_rows(rows),
        "folds": {match_of(f["validation_clip_ids"][0]): sorted(f["validation_clip_ids"]) for f in fold_files},
        "shuffled": json.loads(shuffled_path.read_text(encoding="utf-8")) if shuffled_path.is_file() else None,
    }


def read_tcn_run(folder):
    report = json.loads((folder / "report.json").read_text(encoding="utf-8"))
    if report.get("test_evaluated"):
        print("Note: this TCN run also scored the test split; only its cross-validation is read here.")
    cv = report["models"]["tcn"]["cv"]
    runs = report["models"]["tcn"]["cv_runs"]
    per_seed = []
    for seed in report["config"]["seeds"]:
        scores = []
        for match, epoch in cv["cross_fold_epochs"].items():
            history = next(r["history"] for r in runs if r["held_out"] == match and r["seed"] == seed)
            scores.append(dict(history)[epoch])
        per_seed.append(np.mean(scores))
    with (folder / "cv_predictions.csv").open(encoding="utf-8") as stream:
        rows = [
            {**row, "truth": row["true"], "seed": "ensemble"}
            for row in csv.DictReader(stream)
            if row["model"] == "tcn"
        ]
    folds_by_match = {}
    for row in rows:
        folds_by_match.setdefault(row["match"], []).append(row["clip_id"])
    ensemble = ensemble_from_rows(rows)  # Already seed-averaged probabilities (rounded to 4 dp).
    saved = cv["pooled"]["ensemble"]
    ensemble.update(macro_f1=saved["macro_f1"], accuracy=saved["accuracy"])
    ensemble["per_class"] = {c: saved["per_class"][c]["f1"] for c in classes}
    ensemble["confusion"] = np.array(saved["confusion"])
    return {
        "mode": "other-folds (TCN)",
        "seed_mean": cv["cross_fold_macro_f1"],
        "seed_sd": float(np.std(per_seed)),
        "accuracy": None,
        "per_match": cv["per_fold_macro_f1"],
        "ensemble": ensemble,
        "folds": {m: sorted(v) for m, v in folds_by_match.items()},
        "shuffled": None,
        "control": report["models"].get("control", {}).get("cv", {}).get("cross_fold_macro_f1"),
        "excluded": report["excluded_uncertain"],
    }


other_dir = (ROOT / OTHER_FOLDS_RUN).resolve() if OTHER_FOLDS_RUN else None
tcn_dir = (ROOT / TCN_RUN).resolve() if TCN_RUN else None
protocol = None
if not (other_dir and (other_dir / "summary.json").is_file() and tcn_dir and (tcn_dir / "report.json").is_file()):
    print(f"Need both {other_dir} and {tcn_dir}; this section is skipped.")
    print(
        "Create them with tactic-temporal --epoch-mode other-folds --features (the image cache) "
        "--shuffles 20, and train-tactics --exclude-uncertain --seeds 3 (no --evaluate-test)."
    )
else:
    protocol = {m: read_image_run(other_dir, m) for m in image_models}
    protocol["tcn"] = read_tcn_run(tcn_dir)
    nested = {m: read_image_run(folders[m].parent, m) for m in image_models}
    reference = protocol["mean"]["folds"]
    for name, run in {**protocol, **{f"nested {m}": r for m, r in nested.items()}}.items():
        if run["folds"] != reference:
            raise ValueError(f"{name} does not use the same clips and validation matches.")
    print(f"Reading {other_dir.name} and {tcn_dir.name}: the same 82 clips and four validation matches.")
    print("TCN excluded clips:", protocol["tcn"]["excluded"])
    proto_labels = {"mean": "Baseline (mean)", "thirds": "Thirds", "gru": "GRU", "tcn": "TCN"}
    proto_colours = {**{m: colours[m] for m in image_models}, "tcn": "#CC79A7"}
    order = ["mean", "thirds", "gru", "tcn"]

    print(f"\n{'Model':<18}{'TCN rule: mean over seeds':>28}{'seed ensemble':>16}{'nested (stricter)':>20}")
    for m in order:
        run = protocol[m]
        strict = f"{nested[m]['seed_mean']:.3f}" if m in nested else "not run"
        print(f"{proto_labels[m]:<18}{run['seed_mean']:>20.3f} +/- {run['seed_sd']:.3f}{run['ensemble']['macro_f1']:>16.3f}{strict:>20}")
    print(f"\nTCN time-blind control (same rule): {protocol['tcn']['control']:.3f}")

### Overall

Bars: mean over seeds under the TCN's rule, with the lowest-to-highest seed range for the ResNet models. Diamonds: seed ensemble. Grey dashes: the stricter nested estimate for the ResNet models. Gaps are a rough guide, not a significance test.

In [ ]:
if protocol is None:
    print("Skipped: no TCN-protocol runs.")
else:
    fig, ax = plt.subplots(figsize=(10, 5), layout="constrained")
    for x, m in enumerate(order):
        run = protocol[m]
        ax.bar(x, run["seed_mean"], 0.6, color=proto_colours[m], alpha=0.85, edgecolor="black", linewidth=0.5)
        ax.errorbar(x, run["seed_mean"], yerr=run["seed_sd"], color="black", capsize=4)
        ax.scatter(x, run["ensemble"]["macro_f1"], marker="D", color="white", edgecolor="black", s=70, zorder=4, label="Seed ensemble" if x == 0 else None)
        if m in nested:
            ax.hlines(nested[m]["seed_mean"], x - 0.3, x + 0.3, color="#777777", linestyle="--", linewidth=2, label="Nested (stricter)" if x == 0 else None)
        ax.annotate(f"{run['seed_mean']:.2f}", (x, 0.02), ha="center", color="white", fontsize=10, weight="bold")
    ax.set_xticks(range(len(order)), [proto_labels[m] for m in order])
    ax.set_ylim(0, 1)
    ax.set_ylabel("Macro-F1 (82 training clips, four held-out matches)")
    ax.set_title("Under the TCN's epoch rule (slightly optimistic)")
    ax.legend(loc="upper left")
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)
    plt.show()
    plt.close(fig)

### Per match, per class, and site against split-or-direct

Per match: mean over seeds. Per class and the breakdown: the seed ensemble, pooled over the 82 clips, for every model. **Site right** puts a clip in the right group (A, B or RETAKE); **split or direct right** is for A and B clips whose site was right.

In [ ]:
if protocol is None:
    print("Skipped: no TCN-protocol runs.")
else:
    names = {match_of(f["validation_clip_ids"][0]): fold_names[n] for n, f in folds["mean"].items()}
    print(f"{'Validation match':<26}" + "".join(f"{proto_labels[m]:>17}" for m in order))
    for match, name in names.items():
        print(f"{name:<26}" + "".join(f"{protocol[m]['per_match'][match]:>17.3f}" for m in order))

    print(f"\n{'Class':<10}{'clips':>6}" + "".join(f"{proto_labels[m]:>17}" for m in order))
    for i, label in enumerate(classes):
        clips = int(protocol["mean"]["ensemble"]["confusion"][i].sum())
        print(f"{label:<10}{clips:>6}" + "".join(f"{protocol[m]['ensemble']['per_class'][label]:>17.3f}" for m in order))

    site = {"A_DIRECT": "A", "A_SPLIT": "A", "B_DIRECT": "B", "B_SPLIT": "B", "RETAKE": "RETAKE"}

    def site_split(matrix):
        same_site = [(i, j) for i, t in enumerate(classes) for j, p in enumerate(classes) if site[t] == site[p]]
        attack = [(i, j) for i, j in same_site if site[classes[i]] != "RETAKE"]
        right_site = sum(matrix[i][j] for i, j in same_site) / matrix.sum()
        given = sum(matrix[i][j] for i, j in attack)
        exact = sum(matrix[i][i] for i in range(len(classes)) if site[classes[i]] != "RETAKE")
        return right_site, exact / given, int(given)

    print(f"\n{'Model':<18}{'site right':>12}{'split/direct right, given site':>34}")
    for m in order:
        right_site, split_right, given = site_split(protocol[m]["ensemble"]["confusion"])
        print(f"{proto_labels[m]:<18}{right_site:>12.3f}{split_right:>24.3f} of {given:>3} clips")

    fig, ax = plt.subplots(figsize=(12, 5), layout="constrained")
    x = np.arange(len(classes))
    for offset, m in zip(offsets(len(order)), order, strict=True):
        values = [protocol[m]["ensemble"]["per_class"][label] for label in classes]
        bars = ax.bar(x + offset, values, 0.8 / len(order), color=proto_colours[m], alpha=0.85, edgecolor="black", linewidth=0.5, label=proto_labels[m])
        ax.bar_label(bars, fmt="%.2f", padding=2, fontsize=7)
    ax.set_xticks(x, class_names)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel("F1 (seed ensemble, 82 clips pooled)")
    ax.set_title("Per-class F1 under the TCN's epoch rule")
    ax.legend(ncol=4, loc="upper left")
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)
    plt.show()
    plt.close(fig)

### Shuffled-label check under the TCN's rule

As above, for the three ResNet models refitted under the TCN's rule: each dot is one refit on labels shuffled within each match, the diamond is the real score. The TCN run has no shuffled-label check.

In [ ]:
if protocol is None:
    print("Skipped: no TCN-protocol runs.")
else:
    for m in image_models:
        check = protocol[m]["shuffled"]
        if check is None:
            print(f"{proto_labels[m]}: no shuffled-label check in this run")
            continue
        values = [row["macro_f1"] for row in check["permutations"]]
        print(f"{proto_labels[m]:<18} real {check['real']['macro_f1']:.3f}; shuffled mean {np.mean(values):.3f} (max {max(values):.3f}); {check['at_least_real']} of {check['count']} at or above real")
    print("TCN: no shuffled-label check was run.")

## Keep in mind

**What the saved runs show.** Random thirds scores about the same as the baseline, while thirds is ahead of random thirds on every validation match: this is evidence that time order helps on these development matches, not proof. About two thirds of the gain over the baseline comes from MOUZ vs BetBoom, so with only four matches it is a promising sign rather than a settled result. The GRU is one untuned configuration (the baseline's learning rate, no dropout) on 82 clips; its optimistic score, with the epoch picked on the scored match itself, is also about 0.40, so the epoch choice is not what holds it back.

These are scores on hand-cut clips from the development matches, with every setting chosen while looking at those matches. Nested selection keeps each validation match out of its own model and epoch choice, but four matches is a small sample: the same model's score changes a lot from one validation match to the next, and from seed to seed. Treat a gap smaller than those spreads as a tie. This is a rough guide, not a significance test: the seeds rerun the same four matches. The GRU's design was changed once after seeing development scores (input standardisation), which the other models were not. Under the TCN's epoch rule every score is slightly optimistic, because the other matches' models, which choose the epoch, trained on the scored match; the nested numbers are the stricter estimate. NaVi and Vitality-Falcons remain held out and are never read here. The TCN itself has been scored on the test matches by its author; this notebook reads only its cross-validation.

Small classes have very few clips. The summed confusion matrices count repeated predictions of the same clips, and the per-class bars are means of seed-level F1 scores, not F1 calculated from a pooled matrix.